# Week 5, Day 2: Chunk and store the WWE docs with LangChain

**Lesson:** [week5/day2.ipynb](../../../week5/day2.ipynb), *Dividing documents into chunks*. **About 35 minutes.**

**What you practice here**

- Loading a folder of Markdown files with LangChain's `DirectoryLoader` and `TextLoader`, and tagging each document with its folder as `doc_type`.
- Cutting the documents into overlapping chunks with `RecursiveCharacterTextSplitter` and its `split_documents`.
- Embedding the chunks into a Chroma vector store with `Chroma.from_documents`, starting afresh when one is there.
- Looking inside the store: how many vectors it holds, and how many dimensions each has.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week5/day2", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson loads the Insurellm knowledge base with LangChain's loaders, tags each document with its folder, cuts the
documents into overlapping chunks, embeds the chunks into a Chroma vector store, and looks at what it stored. Do the
same with your WWE documents in `plan/docs`, whose seven folders (`people`, `wrestlemania`, `events`, ...) become the
`doc_type`.

Three of the functions take the LangChain classes they use as their first parameters: `load_documents` takes
`DirectoryLoader` and `TextLoader`, `make_chunks` takes `RecursiveCharacterTextSplitter`, and `create_vectorstore`
takes `Chroma`. `vector_stats` takes the store that `create_vectorstore` returns. The checks run without LangChain's
loaders, splitter and Chroma, so they pass stand-ins shaped like those classes, which record how your code calls them
and work on small folders of their own. In Try it for real you pass LangChain's own classes.

## What each call should give

```python
documents = load_documents(DirectoryLoader, TextLoader, ROOT / "plan" / "docs")
documents[0].metadata
# {'source': '.../plan/docs/<folder>/<file>.md', 'doc_type': '<folder>'}, such as 'people' or 'wrestlemania'

chunks = make_chunks(RecursiveCharacterTextSplitter, documents)
# the chunks of a splitter set to 1,000 characters with an overlap of 200, each with its document's metadata

vectorstore = create_vectorstore(Chroma, embeddings, chunks)
# a Chroma store of the chunks, built with the embeddings and saved in the folder "vector_db"

vector_stats(vectorstore)
# (the number of chunks, 384) with all-MiniLM-L6-v2, whose vectors have 384 numbers
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import glob
import os

### 1 · `load_documents(DirectoryLoader, TextLoader, folder)`

Loads every Markdown file below `folder` with LangChain's loaders, and tags each document with its folder, as the
lesson does. For each folder directly inside `folder` (skip any files there), make one directory loader for that
folder: it finds the `.md` files at any depth below it and reads each with `TextLoader`, as UTF-8. Load its documents,
set each one's `"doc_type"` metadata to the folder's name, such as `people`, and return all the documents in one list.

<details><summary>Hint</summary>

```python
loader = DirectoryLoader(path, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
for doc in loader.load():
    doc.metadata["doc_type"] = os.path.basename(path)
```

</details>

In [ ]:
def load_documents(DirectoryLoader, TextLoader, folder):
    # Write load_documents here, as described above.
    raise NotImplementedError("load_documents is not written yet")

In [ ]:
check("load_documents")

### 2 · `make_chunks(RecursiveCharacterTextSplitter, documents, chunk_size=1000, chunk_overlap=200)`

Cuts the documents into overlapping chunks with LangChain's splitter. Build one splitter with chunks of `chunk_size`
characters that overlap by `chunk_overlap`, passing both by name: its first positional parameter is something else.
Return what its method for documents makes of `documents`, so that every chunk keeps a copy of its document's
metadata, `doc_type` included. The defaults are the lesson's sizes.

<details><summary>Hint</summary>

```python
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
return text_splitter.split_documents(documents)
```

</details>

In [ ]:
def make_chunks(RecursiveCharacterTextSplitter, documents, chunk_size=1000, chunk_overlap=200):
    # Write make_chunks here, as described above.
    raise NotImplementedError("make_chunks is not written yet")

In [ ]:
check("make_chunks")

### 3 · `create_vectorstore(Chroma, embeddings, chunks, db_name="vector_db")`

Embeds the chunks into a Chroma vector store saved in the folder `db_name`, starting afresh. If that folder exists,
first open the store there with the embeddings and delete its collection, as the lesson does, so a rerun does not add
the chunks twice. Then build the store from the chunks with the embeddings, saved in `db_name`, and return it. The
constructor and the builder name the embeddings differently, and a wrong name raises `TypeError`: see the hint.

<details><summary>Hint</summary>

```python
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
return Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
```

</details>

In [ ]:
def create_vectorstore(Chroma, embeddings, chunks, db_name="vector_db"):
    # Write create_vectorstore here, as described above.
    raise NotImplementedError("create_vectorstore is not written yet")

In [ ]:
check("create_vectorstore")

### 4 · `vector_stats(vectorstore)`

Returns how many vectors the store holds and how many numbers each one has, as a pair `(count, dimensions)`. Read the
store's Chroma collection, its `_collection`: count its vectors, then get one vector with the embeddings included,
since Chroma leaves them out unless asked, and take its length. Return `(0, 0)` for an empty store.

<details><summary>Hint</summary>

```python
collection = vectorstore._collection
count = collection.count()
vector = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
```

</details>

In [ ]:
def vector_stats(vectorstore):
    # Write vector_stats here, as described above.
    raise NotImplementedError("vector_stats is not written yet")

In [ ]:
check("vector_stats")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 5, Day 2 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Embedding with `all-MiniLM-L6-v2` runs on your machine and costs nothing; the first run downloads the model, about 90 MB. Days 3 to 5 reopen the store this builds.

1. Measure the knowledge base, as the lesson's first cells do: the number of `.md` files below
   `DOCS = ROOT / "plan" / "docs"` (the setup cell set `ROOT` to your fork's folder), their characters, and their
   tokens with `tiktoken.encoding_for_model("gpt-4.1-nano")`.
2. Load and chunk them with LangChain's own classes: `load_documents(DirectoryLoader, TextLoader, str(DOCS))`, where
   `DOCS` is that docs folder, then `make_chunks(RecursiveCharacterTextSplitter, documents)`. Print how many documents
   and chunks you get, and one chunk.
3. Make `embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` and build the store with
   `create_vectorstore`, in the folder `ROOT / "vector_db"`: at the top of your fork, outside `plan/exercises/`, with
   a name the fork's `.gitignore` already leaves out. Days 3 to 5 reuse it. Print `vector_stats(vectorstore)`.
4. Picture the vectors, as the lesson's Part C does: get the embeddings, documents and metadatas from
   `vectorstore._collection`, give each `doc_type` a color, reduce the vectors to two dimensions with t-SNE
   (`TSNE(n_components=2, random_state=42)`), and plot them with plotly.

<details><summary>Stuck? One way to write it</summary>

```python
import numpy as np
import plotly.graph_objects as go
import tiktoken
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE

DOCS = ROOT / "plan" / "docs"
texts = [path.read_text(encoding="utf-8") for path in DOCS.rglob("*.md")]
encoding = tiktoken.encoding_for_model("gpt-4.1-nano")
print(f"{len(texts)} files, {sum(len(t) for t in texts):,} characters, "
      f"{sum(len(encoding.encode(t)) for t in texts):,} tokens")

documents = load_documents(DirectoryLoader, TextLoader, str(DOCS))
chunks = make_chunks(RecursiveCharacterTextSplitter, documents)
print(f"{len(documents)} documents, {len(chunks)} chunks")
print(chunks[0])

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = create_vectorstore(Chroma, embeddings, chunks, db_name=str(ROOT / "vector_db"))  # git leaves it out
print(vector_stats(vectorstore))

result = vectorstore._collection.get(include=["embeddings", "documents", "metadatas"])
doc_types = [metadata["doc_type"] for metadata in result["metadatas"]]
palette = dict(zip(sorted(set(doc_types)), ["blue", "green", "red", "orange", "purple", "brown", "gray"]))
points = TSNE(n_components=2, random_state=42).fit_transform(np.array(result["embeddings"]))
fig = go.Figure(go.Scatter(x=points[:, 0], y=points[:, 1], mode="markers",
                           marker=dict(size=5, color=[palette[t] for t in doc_types], opacity=0.8),
                           text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, result["documents"])],
                           hoverinfo="text"))
fig.update_layout(title="2D Chroma Vector Store Visualization", width=800, height=600)
fig.show()
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.